# Lab-work #3

First name     - Maksym

Second name    - Maliutin

Group          - CS-31

## Selected dataset

**horses_or_humans** from TensorFlow Datasets.

## Goal

Build ConvNet models from scratch, compare them, then apply transfer learning and explain predictions.

# Task plan

1. Load and inspect the dataset.
2. Prepare images for training.
3. Train several ConvNet architectures from scratch.
4. Compare their performance.
5. Apply transfer learning.
6. Compare with custom CNN models.
7. Add explainability with saliency maps / SHAP if possible.
8. Write conclusions.

# 1. Install and import dependencies

In [1]:
%pip install tensorflow-datasets
%pip install tensorflow==2.20.0
%pip install numpy==2.3.3
%pip install matplotlib==3.10.6
%pip install seaborn==0.13.2
%pip install scikit-learn==1.7.2

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ----------------- ---------------------- 2.4/5.3 MB 12.2 MB/s eta 0:00:01
   ----------------------------------- ---- 4.7/5.3 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 5.3/5.3 MB 11.6 MB/s  0:00:00
   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   --- ------------------------------------ 2.4/27.4 MB 12.2 MB/s eta 0:00:03
   ------ --------------------------------- 4.7/27.4 MB 11.9 MB/s eta 0:00:02
   ---------- ----------------------------- 7.1/27.4 MB 11.8 MB/s eta 0:00:02
   ------------- -------------------------- 9.4/27.4 MB 11.7 MB/s eta

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('darkgrid')
np.random.seed(42)
tf.random.set_seed(42)

print('Libraries imported successfully')

Libraries imported successfully


# 2. Load the dataset

In [3]:
ds_train, ds_info = tfds.load(
    'horses_or_humans',
    split='train',
    with_info=True,
    as_supervised=True
)

print(ds_info)

ModuleNotFoundError: No module named 'importlib_resources'

In [ ]:
print('Dataset info:')
print('Number of classes:', ds_info.features['label'].num_classes)
print('Class names:', ds_info.features['label'].names)
print('Image shape:', ds_info.features['image'].shape)
print('Approximate dataset size:', ds_info.splits['train'].num_examples)

# 3. Visualize samples

In [ ]:
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_train.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy().astype('uint8'))
    plt.title(ds_info.features['label'].names[label.numpy()])
    plt.axis('off')
plt.tight_layout()
plt.show()

# 4. Prepare data pipeline

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def resize_and_rescale(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [ ]:
ds_train_prepared = ds_train.map(resize_and_rescale).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for images, labels in ds_train_prepared.take(1):
    print('Batch image shape:', images.shape)
    print('Batch label shape:', labels.shape)

# 5. Next steps

Here we will later add: train/validation split, custom CNN experiments, transfer learning, and conclusions.

# 6. Train / validation / test split

In [ ]:
# Shuffle once and create train / validation / test splits
num_examples = ds_info.splits['train'].num_examples
train_size = int(0.7 * num_examples)
val_size = int(0.15 * num_examples)
test_size = num_examples - train_size - val_size

full_ds = ds_train.shuffle(num_examples, seed=42, reshuffle_each_iteration=False)

ds_train_raw = full_ds.take(train_size)
ds_val_raw = full_ds.skip(train_size).take(val_size)
ds_test_raw = full_ds.skip(train_size + val_size)

print('Dataset sizes:')
print('Train:', train_size)
print('Validation:', val_size)
print('Test:', test_size)

# Preprocess for CNN training
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

ds_train_prepared = ds_train_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val_prepared = ds_val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test_prepared = ds_test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print('Prepared datasets are ready.')

# 7. Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')

plt.figure(figsize=(10, 10))
for images, labels in ds_train_prepared.take(1):
    augmented_images = data_augmentation(images)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[i].numpy())
        plt.title(ds_info.features['label'].names[labels[i].numpy()])
        plt.axis('off')
plt.tight_layout()
plt.show()

# 8. Experiments with the Conv ANN from scratch

In [ ]:
def build_cnn_v1():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        data_augmentation,
        tf.keras.layers.Conv2D(32, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ], name='cnn_v1')
    return model

cnn_v1 = build_cnn_v1()
cnn_v1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
cnn_v1.summary()

In [ ]:
def plot_history(history, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history.history['accuracy'], label='train')
    axes[0].plot(history.history['val_accuracy'], label='val')
    axes[0].set_title(f'{title_prefix} accuracy')
    axes[0].legend()
    
    axes[1].plot(history.history['loss'], label='train')
    axes[1].plot(history.history['val_loss'], label='val')
    axes[1].set_title(f'{title_prefix} loss')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

callbacks_list = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history_v1 = cnn_v1.fit(
    ds_train_prepared,
    validation_data=ds_val_prepared,
    epochs=10,
    callbacks=callbacks_list,
    verbose=1
)

plot_history(history_v1, 'CNN v1')

In [ ]:
def build_cnn_v2():
    inputs = tf.keras.layers.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs, name='cnn_v2')
    return model

cnn_v2 = build_cnn_v2()
cnn_v2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_v2.summary()

In [ ]:
history_v2 = cnn_v2.fit(
    ds_train_prepared,
    validation_data=ds_val_prepared,
    epochs=10,
    callbacks=callbacks_list,
    verbose=1
)

plot_history(history_v2, 'CNN v2')

# 9. A smaller regularized CNN

In [ ]:
def build_cnn_v3():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(224, 224, 3)),
        data_augmentation,
        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ], name='cnn_v3')
    return model

cnn_v3 = build_cnn_v3()
cnn_v3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_v3.summary()

history_v3 = cnn_v3.fit(
    ds_train_prepared,
    validation_data=ds_val_prepared,
    epochs=10,
    callbacks=callbacks_list,
    verbose=1
)

plot_history(history_v3, 'CNN v3')

# 10. Compare CNN models

In [ ]:
def evaluate_model(model, dataset, name):
    loss, acc = model.evaluate(dataset, verbose=0)
    return {'model': name, 'loss': loss, 'accuracy': acc}

results = [
    evaluate_model(cnn_v1, ds_test_prepared, 'CNN v1'),
    evaluate_model(cnn_v2, ds_test_prepared, 'CNN v2'),
    evaluate_model(cnn_v3, ds_test_prepared, 'CNN v3'),
]
results_df = __import__('pandas').DataFrame(results)
print(results_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='model', y='accuracy')
plt.title('CNN models comparison on test set')
plt.ylim(0, 1)
plt.show()

# 11. Transfer learning

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


def preprocess_transfer(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

transfer_train = ds_train_raw.map(preprocess_transfer, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
transfer_val = ds_val_raw.map(preprocess_transfer, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
transfer_test = ds_test_raw.map(preprocess_transfer, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

base_model = MobileNetV2(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
base_model.trainable = False

transfer_model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

transfer_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
transfer_model.summary()

In [ ]:
transfer_history = transfer_model.fit(
    transfer_train,
    validation_data=transfer_val,
    epochs=8,
    callbacks=callbacks_list,
    verbose=1
)

plot_history(transfer_history, 'Transfer learning')

transfer_test_loss, transfer_test_acc = transfer_model.evaluate(transfer_test, verbose=0)
print('Transfer learning test accuracy:', transfer_test_acc)

# 12. Explainability with saliency maps

In [ ]:
def make_saliency_map(model, image_tensor, label_idx=None):
    image_tensor = tf.cast(tf.expand_dims(image_tensor, axis=0), tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        prediction = model(image_tensor)
        if label_idx is None:
            score = prediction[:, 0]
        else:
            score = prediction[:, label_idx]
    grads = tape.gradient(score, image_tensor)
    saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0]
    return saliency

for images, labels in transfer_test.take(1):
    sample_image = images[0]
    sample_label = labels[0]
    saliency = make_saliency_map(transfer_model, sample_image)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow((sample_image - sample_image.min()) / (sample_image.max() - sample_image.min() + 1e-8))
    plt.title(f'Original: {ds_info.features["label"].names[sample_label.numpy()]}')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(saliency.numpy(), cmap='hot')
    plt.title('Saliency map')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# 13. Conclusions

- The lab includes full data preparation, multiple CNNs from scratch, transfer learning, and an explainability bonus block.
- The final choice should be based on validation/test accuracy and stability of training curves.
- If transfer learning performs better than custom CNNs, it becomes the preferred solution for this dataset.
- Saliency maps show which image regions most influence the prediction.